# Match eval_detail results across two test runs\nJoins on `(prompt, actual_label)` so rows from different model runs can be compared side-by-side.

In [ ]:
# ── File paths ──────────────────────────────────────────────────────────────
LEFT_CSV  = "data/test1/eval_detail.csv"      # test run 1
RIGHT_CSV = "data/test2/eval_detail_2.csv"    # test run 2
OUTPUT_CSV = "data/matched_eval.csv"

JOIN_TYPE = "inner"   # inner | left | right | outer

In [ ]:
import pandas as pd
from pathlib import Path

JOIN_COLS = ["prompt", "actual_label"]

def load(path):
    df = pd.read_csv(path)
    df = df.dropna(how="all").reset_index(drop=True)
    df.columns = [c.strip() for c in df.columns]
    df = df.loc[:, df.columns != ""]
    df = df.loc[:, ~df.columns.str.match(r"^Unnamed")]
    for col in JOIN_COLS:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
    return df

left  = load(LEFT_CSV)
right = load(RIGHT_CSV)

print(f"Left  rows (after cleaning): {len(left)}")
print(f"Right rows (after cleaning): {len(right)}")
print(f"\nLeft  columns:  {list(left.columns)}")
print(f"Right columns:  {list(right.columns)}")

In [ ]:
left_rename  = {c: f"{c}_t1" for c in left.columns  if c not in JOIN_COLS}
right_rename = {c: f"{c}_t2" for c in right.columns if c not in JOIN_COLS}

merged = pd.merge(
    left.rename(columns=left_rename),
    right.rename(columns=right_rename),
    on=JOIN_COLS,
    how=JOIN_TYPE,
)

print(f"Matched rows:    {len(merged)}")
print(f"Unmatched left:  {len(left)  - len(merged)}" if JOIN_TYPE == "inner" else "")
print(f"Unmatched right: {len(right) - len(merged)}" if JOIN_TYPE == "inner" else "")

merged.head(3)

In [ ]:
merged